# Booking propensity mode

This notebook models the relationship between fighter merit and main event usage. The two axes place a fighter by competitive record and by public profile; this asks how those two merit signals relate to whether the promotion has booked a fighter as a headliner, and reads the departures from that relationship as the actionable output.

The prediction itself is deliberately unsurprising: high rated, high profile fighters headline more, and a model that captures that is confirming the promotion books largely on merit. The product is the residual. A fighter whose merit predicts headlining but who has not headlined is under booked relative to merit; a fighter booked as a headliner whose merit does not predict it is over booked. The under booked set is the framework's headline applicability output, the fighters a matchmaker could elevate on the evidence of demonstrated merit.

The model is descriptive, not causal. It learns the historical relationship between merit and booking, a relationship formed overwhelmingly under the pay-per-view era, and flags who sits off it. It does not predict who *should* headline in any normative sense, nor who *will* draw interest if elevated; those readings are supplied in interpretation, argued through the post pay-per-view lens, not asserted by the model. Because the model learns the historical pattern including any bias in it, demographic questions are put to the residuals, never to the features.

Two limitations are stated here so they travel with every number below. The target is binary (headlined or not), so it treats all main events as equivalent; headlining a Fight Night and headlining a numbered card count the same, and the residual should be read with that in mind (a two tier target is recorded as future work). And "under booked relative to merit" is only as good as the two axes, so a residual is a flag for inspection, not a verdict; a fighter may be under booked for a real reason the axes do not capture.

In [ ]:
# BLOCK 1: setup

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

# data location (portable across Drive and a local repo checkout)
try:
    from google.colab import drive
    drive.mount('/content/drive')
except (ImportError, ModuleNotFoundError):
    pass  # not in Colab

DRIVE_DIR = Path('/content/drive/MyDrive/Masters in Artificial Intelligence Applied to Sport/'
                 'Masters Final Project/Pugnator mapper valorem/EDA/Code Outputs')
OUTPUT_DIR = DRIVE_DIR if DRIVE_DIR.exists() else Path('./data')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Using data directory: {OUTPUT_DIR}')

Mounted at /content/drive


## Section 1: The modelling population

The model is fit on Established and Provisional fighters with a multi-source public profile. The Unreliable tier is excluded because there a fighter with no main event is almost always too new to have been booked as one rather than passed over, so their residual would not carry the intended meaning. Single-source profiles are excluded because the merit reading is too thin to support a residual. This leaves a population where both merit axes are reliable and a zero main-event rate is informative.

The target is binary: whether a fighter has ever been the main event fighter. The features are the two merit axes only. No career-stage or demographic variable enters the model, so the residual means departure from the merit to booking relationship and nothing else; career stage is carried alongside the residual as context rather than modelled.

In [ ]:
# BLOCK 2: assemble the modelling population and the binary target

prof = pd.read_parquet(OUTPUT_DIR / 'public_profile_axis.parquet')      # public_profile, main_event_rate, profile_reliability
glicko = pd.read_parquet(OUTPUT_DIR / 'glicko_current_continuous.parquet')[['FIGHTER', 'RATING', 'TIER', 'N_FIGHTS']]
career = pd.read_parquet(OUTPUT_DIR / 'career_signals.parquet')[['FIGHTER', 'main_events']]

df = prof.merge(glicko, on='FIGHTER').merge(career, on='FIGHTER', how='left')

# reliable both sides: Established or Provisional rating, multi source profile
pop = df[df['TIER'].isin(['Established', 'Provisional']) & (df['profile_reliability'] == 'multi_source')].copy()

# binary target: has the fighter ever been the main event
pop['headlined'] = (pop['main_event_rate'] > 0).astype(int)

print(f'modelling population: {len(pop)} fighters')
print(f'headlined: {pop.headlined.sum()} ({100 * pop.headlined.mean():.0f}%), not headlined: {(pop.headlined == 0).sum()}')

modelling population: 439 fighters
headlined: 185 (42%), not headlined: 254


The modelling population is 439 fighters with reliable merit on both axes. The classes are reasonably balanced: 185 (42%) have headlined at least once, 254 have not. A near even split means the model is learning a genuine distinction rather than a majority class default, and the residuals on both sides rest on adequate numbers.

## Section 2: The model

A logistic regression is the primary model: with two features its coefficients are directly interpretable and its probability output gives a clean residual. A gradient boosted classifier is fitted alongside as a robustness check, to confirm the merit to booking relationship is not an artefact of the linear form. Features are standardised so the logistic coefficients are comparable across the two axes.

Probabilities are produced out of fold by five fold cross validation, so each fighter's predicted probability, and therefore their residual, comes from a model that did not see them during fitting. The performance figures (AUC) establish that merit predicts booking well; they are not the point of the notebook and are not optimised.

In [ ]:
# BLOCK 3: fit the primary and robustness models with out of fold probabilities
FEATURES = ['RATING', 'public_profile']
X = StandardScaler().fit_transform(pop[FEATURES].to_numpy())
y = pop['headlined'].to_numpy()
cv = StratifiedKFold(5, shuffle=True, random_state=42)

logit = LogisticRegression(max_iter=1000)
gbm = GradientBoostingClassifier(random_state=42)

# out of fold predicted probability of having headlined, so residuals are honest
p_logit = cross_val_predict(logit, X, y, cv=cv, method='predict_proba')[:, 1]
p_gbm = cross_val_predict(gbm, X, y, cv=cv, method='predict_proba')[:, 1]

print(f'AUC  logistic = {roc_auc_score(y, p_logit):.3f}')
print(f'AUC  gradient-boosted = {roc_auc_score(y, p_gbm):.3f}')
print(f'agreement of the two probability vectors: r = {np.corrcoef(p_logit, p_gbm)[0, 1]:.3f}')

# standardised coefficients from the primary model, refit on all rows for reporting
logit.fit(X, y)
print(f'\nstandardised coefficients: RATING = {logit.coef_[0][0]:.2f}, public_profile = {logit.coef_[0][1]:.2f}')

AUC  logistic = 0.881
AUC  gradient-boosted = 0.862
agreement of the two probability vectors: r = 0.882

standardised coefficients: RATING = 0.86, public_profile = 1.46


Merit predicts booking well. The logistic model reaches an AUC of 0.881 and the gradient-boosted model 0.862; the two probability vectors agree closely (r = 0.882), so the relationship is not an artefact of the linear form and the simpler, interpretable model is retained as primary. The standardised coefficients show public profile carrying more weight than competitive rating (1.46 against 0.86): headlining tracks public attention more strongly than it tracks demonstrated ability, which is the promotion booking on draw as much as on merit, and is consistent with the framework's premise that the two are distinct.

## Section 3: Residuals, the actionable output

The residual is the observed outcome minus the predicted probability. A fighter the model expects to have headlined (high predicted probability) who has not (outcome zero) has a large negative residual and is under booked relative to merit; a fighter who has headlined (outcome one) whose merit predicts a low probability has a large positive residual and is over booked.

Two context columns travel with the residual and are not inputs to it. On the under booked side, fight count distinguishes a high merit prospect who is early in their run from a longer tenured fighter who has been passed over; both are genuine under booked signals but a matchmaker would act on them differently. On the over booked side, the main event count distinguishes a one time booking, often a short notice replacement, from a fighter the promotion has repeatedly featured. The under booked signal is the robust one and leads; the over booked signal is secondary and read with its counts, since a single booking can place a low merit fighter at the top of it.

In [ ]:
# BLOCK 4: residuals and the two ranked lists

pop['pred'] = p_logit
pop['residual'] = pop['headlined'] - pop['pred']   # negative: merit predicts headliner, has not

residuals = pop[['FIGHTER', 'RATING', 'public_profile', 'TIER',
                 'N_FIGHTS', 'main_events', 'headlined', 'pred', 'residual']].copy()
residuals.to_parquet(OUTPUT_DIR / 'booking_residuals.parquet', index=False)

# under booked: never headlined, ranked by how strongly merit predicted they would; fight count as context
under = residuals[residuals['headlined'] == 0].nsmallest(15, 'residual')
print('UNDER-BOOKED (merit predicts headliner, has not; N_FIGHTS = tenure context):')
print(under[['FIGHTER', 'RATING', 'public_profile', 'N_FIGHTS', 'pred', 'residual']].round(2).to_string(index=False))

# over booked: has headlined, ranked by how little merit predicted it; main event count as context
over = residuals[residuals['headlined'] == 1].nlargest(10, 'residual')
print('\nOVER-BOOKED (headlined despite low merit; main_events = how many times):')
print(over[['FIGHTER', 'RATING', 'public_profile', 'main_events', 'pred', 'residual']].round(2).to_string(index=False))

print(f'\nsaved booking_residuals.parquet ({len(residuals)} fighters)')

UNDER-BOOKED (merit predicts headliner, has not; N_FIGHTS = tenure context):
            FIGHTER  RATING  public_profile  N_FIGHTS  pred  residual
  Shavkat Rakhmonov 2050.95            1.22         7  0.98     -0.98
     Kayla Harrison 1899.01            1.52         3  0.96     -0.96
         Josh Hokit 1928.26            1.21         4  0.94     -0.94
       Michael Page 1874.95            1.18         5  0.92     -0.92
          Bo Nickal 1797.50            1.32         7  0.91     -0.91
         Joshua Van 1827.22            1.33        11  0.90     -0.90
     Mauricio Ruffy 1832.84            1.07         6  0.88     -0.88
Waldo Cortes Acosta 1837.82            0.85        13  0.84     -0.84
    Shara Magomedov 1799.73            1.06         7  0.83     -0.83
     Tatiana Suarez 1884.22            0.67        10  0.81     -0.81
     Bryce Mitchell 1802.81            0.97        13  0.81     -0.81
      Natalia Silva 1881.50            0.43         8  0.80     -0.80
       Joel A

The under booked list reads as a mix of two kinds of fighter, separated by the tenure context rather than by the residual alone. Several are high merit prospects still early in their run (Shavkat Rakhmonov at 7 fights, Kayla Harrison at 3, Bo Nickal at 7), where a near certain predicted probability and no headline reflects a fighter the promotion has not yet elevated rather than one it has passed over. Others carry longer records without a headline (Waldo Cortes Acosta and Bryce Mitchell at 13 fights, Joel Alvarez and Aiemann Zahabi at 11), where the same signal points to a fighter overlooked despite tenure. Both are genuine under-booked flags, but a matchmaker would read the two differently, which is why fight count travels alongside the residual.

The over booked list is dominated by single bookings: nine of the ten fighters have a main_events count of one, several of them plausibly short notice replacements (Ion Cutelaba, HyunSung Park). This is the fragility the design anticipates, one booking can place a low merit fighter at the top of the over booked ranking, so the over booked side is read with its counts and treated as secondary to the under booked signal, which is the robust one.

## Section 4: Feature attribution

With two features the standardised logistic coefficients are the attribution: they show how much each merit axis moves the predicted probability of having headlined, on a common scale. This is reported as transparency on how the model reaches its prediction, not as a separate result.

In [ ]:
# BLOCK 5: attribution from the standardised coefficients
coef = pd.Series(logit.coef_[0], index=FEATURES).sort_values(key=abs, ascending=False)
print('standardised logistic coefficients (contribution to the log-odds of having headlined):')
for name, val in coef.items():
    print(f'  {name:16} {val:+.2f}')

standardised logistic coefficients (contribution to the log-odds of having headlined):
  public_profile   +1.46
  RATING           +0.86


Public profile dominates the attribution, contributing +1.46 to the log-odds of having headlined against +0.86 for competitive rating. In plain terms, the promotion's headline bookings track public attention more strongly than competitive standing: a fighter's draw moves their booking probability further than their record does. This is a descriptive reading of historical booking under the pay-per-view era, not a prescription; it is the empirical basis for the framework's premise that a fighter can be strong on merit yet under booked because their public profile has not yet caught up.

## Section 5: Pre registration of the retrospective validation

This section is written before the validation is run, so the hypothesis is fixed in advance of any result.

The residual claims to identify fighters whose merit warrants headlining before the promotion has acted on it. If that claim holds, the under booked fighters should show larger subsequent gains in public profile than comparable fighters who were not flagged. The test, to be run once the two axes are reconstructed at an earlier cutoff:

- **Cutoff.** Reconstruct the competitive and public profile axes as they stood at an earlier date (the competitive axis from the pre fight rating history, the public profile from the pageview series re windowed to that date and a GDELT query ending there). Refit the booking propensity model on the reconstructed axes to obtain residuals as of the cutoff.
- **Hypothesis (directional, fixed in advance).** Among fighters with a reliable rating at the cutoff, those flagged as under booked (large negative residual) will show a greater increase in public profile between the cutoff and the freeze than fighters matched on competitive rating at the cutoff who were not flagged.
- **Design.** Matched comparison on competitive rating at the cutoff, so the two groups differ in their booking residual and not in demonstrated ability. Matching on ability also differences out any era wide shift in profile between the cutoff and the freeze, since it affects both groups alike.
- **Reading.** This is a directional, leading indicator check, not causal proof. A positive result is evidence that the residual precedes real movement; a null result is the honest finding that the residual is contemporaneous rather than leading, and is reported as such. The named case studies are drawn only from fighters with the tenure to have been booked, so a rising prospect who was always going to headline is not counted as a validation of the flag.

The reconstruction and this validation are recorded as the next step; the hypothesis is pre-registered here so it stands fixed in advance of any result.

## Conclusion

The booking propensity model confirms that the promotion books main events largely on merit: the two axes predict whether a fighter has headlined with an AUC of 0.881, and a gradient-boosted model agrees closely, so the relationship is real rather than an artefact of the linear form. Public profile carries more weight than competitive rating in that relationship (standardised coefficients 1.46 against 0.86), which is the promotion booking on draw as much as on demonstrated ability.

The model's value is not the prediction but the departures from it. The residuals surface a concrete under-booked list, fighters whose merit predicts headlining but who have not been booked as one, separating high-merit prospects early in their run from longer tenured fighters who have been passed over. This is the framework's headline applicability output: a merit-grounded shortlist a matchmaker could act on, read through the post-PPV lens where a fighter's demonstrated value and their current booking can diverge. The over-booked side is retained as a secondary, more fragile signal, dominated by single bookings and read with its counts.

The model is descriptive of historical booking under the pay-per-view era, not causal or normative; the residual is a flag for inspection, not a verdict. Its leading-indicator claim, that an under booked flag precedes real movement in profile, is pre-registered below for retrospective validation once the axes are reconstructed at an earlier cutoff.